# Module 9: Deploy Your Agent on the Inference You Own

In Module 8 you benchmarked your server and wrote down an operating point you could defend: a model, a concurrency, and the throughput, latency, and cost that justify it. You tuned the inference layer until it was yours. Now you put an agent on top of it. This is where the whole workshop pays off. You install [kagent](https://kagent.dev/) into your cluster, point it at the in-cluster [vLLM](https://docs.vllm.ai) you have been tuning, deploy an Akamai Cloud solutions architect agent as a Kubernetes resource, and talk to it. The brain answering every turn is the server you own, on your GPU, not a rented API.

## Learning objectives
- Install kagent into your cluster and confirm the controller is running
- Build the cross-namespace FQDN that lets an agent in one namespace reach your vLLM in another
- Open the default-deny boundary with a `NetworkPolicy` that allows only kagent to reach vLLM
- Point a kagent `ModelConfig` at your in-cluster vLLM using the OpenAI provider
- Deploy an `Agent` resource carrying a scoped system prompt and confirm it answers from your vLLM
- Test the agent in scope and out of scope, and see it route honestly instead of guessing

## Prerequisites
- Finished the inference labs, especially `05_optimize_the_server`, with a working vLLM Service in your namespace
- A live cluster with `helm`, `kubectl`, and credentials that can create resources in the `kagent` namespace
- The vLLM you point at must be served with tool calling enabled (`--enable-auto-tool-choice` and a matching `--tool-call-parser`)
- Optional: the `kagent` CLI installed (`curl -sfL https://kagent.dev/install.sh | bash`) to run the invoke step
- About 12 minutes

References: [kagent](https://kagent.dev/) &middot; [kagent install lab](https://labeveryday.github.io/learn-k8s/07-kagent/lab-01-install-kagent/) &middot; [vLLM tool calling](https://docs.vllm.ai/en/latest/features/tool_calling.html) &middot; [Kubernetes NetworkPolicy](https://kubernetes.io/docs/concepts/services-networking/network-policies/) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/)

## Agent on owned inference design basics

An agent is just a client of a model. The question this module answers is where that model lives. kagent runs agents as Kubernetes custom resources, and two of them carry the whole story.

- **ModelConfig.** Tells kagent where the model is and how to talk to it. Yours uses the OpenAI provider, because vLLM is OpenAI-compatible, and a `baseUrl` that points at your in-cluster vLLM Service. The agent's brain is the server you tuned.
- **Agent.** Defines the assistant: its system prompt, the `ModelConfig` it uses, and any tools. Apply it and kagent reconciles a running agent you can invoke.

There is one boundary to cross. kagent runs in the `kagent` namespace and your vLLM runs in yours, so a bare Service name like `vllm` will not resolve across the gap. You qualify it to the cross-namespace FQDN `vllm.<your-namespace>.svc.cluster.local`, and you open the workshop's default-deny `NetworkPolicy` with one rule that allows only kagent to reach vLLM. Nothing else changes.

![A kagent Agent in the kagent namespace reaches a ModelConfig with the OpenAI provider, which calls your in-cluster vLLM on your GPU across namespaces over the cluster FQDN, allowed by a NetworkPolicy](images/09_agents_on_k8s_architecture.png)

## 1. Setup

This module reads its connection details from `common/config.py` and drives `kubectl` and `helm` against a live cluster. Install the one Python dependency the OpenAI-compatible client needs. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q "openai>=1.40"

## 2. Resolve the in-cluster vLLM URL

`get_settings()` reads your `NAMESPACE`, `VLLM_HOST`, and `MODEL_NAME`. The agent lives in the `kagent` namespace, so a bare Service name resolves to nothing for it. The `agent_reachable_url` helper qualifies a bare name to the cross-namespace FQDN `<name>.<namespace>.svc.cluster.local`, and leaves an external URL untouched. The agent's `ModelConfig` uses the value it prints.

In [ ]:
# Setup: settings and the in-cluster vLLM base URL the agent will use.
import os, sys, subprocess, textwrap
from urllib.parse import urlparse
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings

settings = get_settings()
NS = settings.namespace
VLLM_URL = settings.vllm_host          # e.g. http://vllm:8000/v1
MODEL = settings.model_name

# kagent runs in its own `kagent` namespace, while your vLLM Service lives in
# NS. A bare in-cluster name like `vllm` only resolves inside its own namespace,
# so qualify it to a cross-namespace FQDN for the agent. An external URL (an IP
# or a domain) already resolves from anywhere and is left unchanged.
def agent_reachable_url(url, ns):
    u = urlparse(url)
    host = u.hostname or ""
    if host and "." not in host and host != "localhost":
        port = f":{u.port}" if u.port else ""
        return f"{u.scheme}://{host}.{ns}.svc.cluster.local{port}{u.path}"
    return url

AGENT_VLLM_URL = agent_reachable_url(VLLM_URL, NS)

print("namespace      :", NS)
print("vllm url       :", VLLM_URL)
print("agent vllm url :", AGENT_VLLM_URL)
print("model          :", MODEL)

**What you should see:** your namespace, the bare in-cluster vLLM URL, and the qualified agent URL ending in `.svc.cluster.local`. The agent's `ModelConfig` uses exactly that qualified value, so it talks to the server you have been tuning.

## 3. Install kagent

Install the CRDs first, then the controller. Order is load-bearing: the controller crash-loops if the `Agent` and `ModelConfig` types do not exist yet. Both releases come from the kagent OCI helm registry. This module targets the `kagent.dev/v1alpha2` API; field names change between versions, so confirm against your installed CRDs with `kubectl explain modelconfig.spec` and `kubectl explain agent.spec` if an apply is rejected.

In [ ]:
# Requires a live cluster with helm.
# Install kagent CRDs first, then the controller. Order matters.
subprocess.run([
    "helm", "upgrade", "-i", "kagent-crds",
    "oci://ghcr.io/kagent-dev/kagent/helm/kagent-crds",
    "--namespace", "kagent", "--create-namespace",
], check=True)

subprocess.run([
    "helm", "upgrade", "-i", "kagent",
    "oci://ghcr.io/kagent-dev/kagent/helm/kagent",
    "--namespace", "kagent",
], check=True)

print("kagent installed; check the controller pod:")
subprocess.run(["kubectl", "get", "pods", "-n", "kagent"])

**What you should see:** two successful helm releases and a `kubectl get pods` showing the kagent controller `Running` in the `kagent` namespace. If the controller is crash-looping, the CRDs did not install first; re-run the CRDs release and restart the controller.

## 4. Write the agent's persona

This agent is an Akamai Cloud solutions architect assistant. The system prompt sets its scope so it stays useful and honest about what it does not cover.

- **In scope:** Akamai Cloud Compute (Linodes), LKE, Object Storage, Cloud networking (VPC, NodeBalancer, Cloud Firewall), GPUs, and AI inference (vLLM, model serving, agents). Tactical, developer to developer.
- **Out of scope, and it says so:** Akamai CDN, Akamai security products, and edge compute (EdgeWorkers, EdgeKV). It points you to the right team instead of guessing.

A nice touch: it can tell you which model and endpoint it runs on, which reinforces that it lives on self-hosted inference.

In [ ]:
# Write the system prompt to a file so the manifest can embed it.
SYSTEM_PROMPT = textwrap.dedent(f'''
    You are the Akamai Cloud Solutions Architect agent. You help developers with
    Akamai Cloud and the Kubernetes cluster you run in. You are tactical and
    concise, developer to developer.

    In scope: Akamai Cloud Compute (Linodes), LKE (Linode Kubernetes Engine),
    Object Storage, Cloud networking (VPC, NodeBalancer, Cloud Firewall), GPUs,
    and AI inference (vLLM, model serving, agents).

    Out of scope: Akamai CDN, Akamai security products, and edge compute
    (EdgeWorkers, EdgeKV). If asked about these, say they are out of your scope
    and point the user to the right Akamai team. Do not guess.

    You run on self-hosted inference: model {MODEL} served by vLLM at {VLLM_URL}.
    If asked what powers you, say so.
''').strip()

with open("system_prompt.txt", "w") as f:
    f.write(SYSTEM_PROMPT)
print(SYSTEM_PROMPT)

**What you should see:** the full system prompt printed, with your real model and endpoint folded into the last line. It is saved to a file so the next steps can embed it into the Agent manifest.

## 5. Point kagent at your vLLM

Three things happen here. First, the `NetworkPolicy` that lets kagent reach vLLM across the namespace boundary: the workshop runs default-deny, so without this allow the agent cannot call the model. It permits only the `kagent` namespace to reach pods labeled `app: vllm` on port 8000, and nothing else. Second, a placeholder API key Secret, because the OpenAI provider expects a key and vLLM does not enforce one. Third, the `ModelConfig` itself, with the OpenAI provider and your qualified `baseUrl`. Nothing secret goes in the manifest.

In [ ]:
# Requires a live cluster.
# 1) Let the agent reach your vLLM. kagent runs in the `kagent` namespace, and
#    the workshop's default-deny NetworkPolicy blocks cross-namespace traffic,
#    so without this allow the agent cannot call the model. Idempotent; harmless
#    if your environment has no default-deny.
netpol = f'''
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-kagent-to-vllm
  namespace: {NS}
spec:
  podSelector:
    matchLabels:
      app: vllm
  policyTypes: [Ingress]
  ingress:
  - from:
    - namespaceSelector:
        matchLabels:
          kubernetes.io/metadata.name: kagent
    ports:
    - port: 8000
      protocol: TCP
'''
subprocess.run(["kubectl", "apply", "-f", "-"], input=netpol, text=True, check=True)

# 2) Placeholder API key Secret (vLLM does not enforce one) and a ModelConfig
#    that points kagent at your in-cluster vLLM endpoint.
#    Note: kagent's runtime always sends tool-calling requests, so the vLLM you
#    point at must be served with --enable-auto-tool-choice and a matching
#    --tool-call-parser (hermes for Qwen). Without them the agent call fails 400.
secret_yaml = subprocess.run([
    "kubectl", "create", "secret", "generic", "vllm-key",
    "--from-literal=OPENAI_API_KEY=not-needed",
    "-n", "kagent", "--dry-run=client", "-o", "yaml",
], check=True, capture_output=True, text=True).stdout
subprocess.run(["kubectl", "apply", "-f", "-"], input=secret_yaml, text=True, check=True)

modelconfig = f'''
apiVersion: kagent.dev/v1alpha2
kind: ModelConfig
metadata:
  name: vllm
  namespace: kagent
spec:
  provider: OpenAI
  model: "{MODEL}"
  apiKeySecret: vllm-key
  apiKeySecretKey: OPENAI_API_KEY
  openAI:
    baseUrl: "{AGENT_VLLM_URL}"
'''
subprocess.run(["kubectl", "apply", "-f", "-"], input=modelconfig, text=True, check=True)
print("ModelConfig applied, pointing at", AGENT_VLLM_URL)

**What you should see:** the `NetworkPolicy`, the Secret, and the `ModelConfig` applied, with a confirmation line naming the qualified URL. If the apply is rejected on a field name, run `kubectl explain modelconfig.spec` against your installed CRDs and adjust; the provider config block name in particular can differ between kagent versions.

## 6. Create the agent

The `Agent` resource ties the persona to the model. It references the `ModelConfig` by name and carries the system prompt as a YAML block scalar. The indentation is load-bearing: the prompt body must sit deeper than the `systemMessage:` key, or `kubectl` rejects the manifest. No tools yet; you can add a read-only cluster tool or an Akamai docs lookup later through a `RemoteMCPServer`, but the core agent works without them.

In [ ]:
# Requires a live cluster.
# Apply the Agent, embedding the system prompt from the file.
with open("system_prompt.txt") as f:
    sys_msg = f.read()

# Indent the prompt for the YAML block scalar. The body must sit deeper than the
# `systemMessage:` key (6 spaces here, under spec.declarative), or kubectl
# rejects the manifest as invalid YAML.
indented = "\n".join("      " + line for line in sys_msg.splitlines())
agent = f'''
apiVersion: kagent.dev/v1alpha2
kind: Agent
metadata:
  name: akamai-sa-agent
  namespace: kagent
spec:
  description: "Akamai Cloud solutions architect assistant, running on self-hosted vLLM."
  type: Declarative
  declarative:
    modelConfig: vllm
    systemMessage: |
{indented}
    tools: []
'''
subprocess.run(["kubectl", "apply", "-f", "-"], input=agent, text=True, check=True)
subprocess.run(["kubectl", "get", "agent", "-n", "kagent"])
print("agent applied")

**What you should see:** the Agent applied and listed by `kubectl get agent`. It may take a moment to become ready while kagent reconciles it into a running runtime pod.

## 7. Talk to it

Test with `kagent invoke`. The CLI sends your message to the agent, which calls your vLLM for the answer. It reaches the controller API on `localhost:8083`, so you port-forward it first, wait for the agent runtime to be ready, then ask one question in scope and one out of scope. kagent 0.9+ takes `--agent` and `--task` and returns an A2A task as JSON, so the helper pulls the text out of the response.

In [ ]:
# Requires a live cluster and the kagent CLI (see https://kagent.dev to install).
# Wait for the agent runtime to be ready, then talk to it. The CLI reaches the
# controller API on localhost:8083, so port-forward it first. kagent 0.9+ takes
# --agent/--task; invoke returns an A2A task as JSON, so we pull out the text.
import json, time, atexit

# The Agent applied above spins up a runtime pod; give it a moment to be ready.
subprocess.run(["kubectl", "wait", "--for=condition=Ready",
                "agent/akamai-sa-agent", "-n", "kagent", "--timeout=180s"], check=False)

pf = subprocess.Popen(
    ["kubectl", "port-forward", "svc/kagent-controller", "8083:8083", "-n", "kagent"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
atexit.register(pf.terminate)
time.sleep(5)  # let the port-forward establish

def ask(task):
    out = subprocess.run(
        ["kagent", "invoke", "--agent", "akamai-sa-agent", "--task", task, "-n", "kagent"],
        capture_output=True, text=True,
    )
    try:
        d = json.loads(out.stdout)
        texts = [p["text"] for m in d.get("history", []) if m.get("role") == "agent"
                 for p in m.get("parts", []) if p.get("kind") == "text"]
        print(texts[-1] if texts else out.stdout.strip())
    except Exception:
        print(out.stdout.strip() or out.stderr.strip())

print("=== in scope ===")
ask("How do I create an LKE cluster with a GPU node pool?")
print("\n=== out of scope ===")
ask("How do I write an EdgeWorker for the CDN?")

pf.terminate()

**What you should see:** a tactical answer about LKE and GPU node pools for the first question, and a polite "that is out of my scope, talk to the edge compute team" for the second. Both answers came from your vLLM, on your GPU. If the CLI is not installed, install it with `curl -sfL https://kagent.dev/install.sh | bash`, or port-forward the kagent service and curl its API.

## Things to know

- **Tool calling is not optional for kagent.** kagent's runtime always sends tool-calling requests, so the vLLM you point at must be served with `--enable-auto-tool-choice` and a matching `--tool-call-parser` (`hermes` for Qwen). Without them the agent call fails with a 400, even for a plain question.
- **The FQDN is the whole cross-namespace trick.** A bare Service name resolves only inside its own namespace. From the `kagent` namespace your vLLM is `vllm.<your-namespace>.svc.cluster.local`. The `agent_reachable_url` helper builds that for you and leaves external URLs alone.
- **Default-deny means you grant reachability explicitly.** The `allow-kagent-to-vllm` `NetworkPolicy` is what lets kagent reach the model at all. It permits only the `kagent` namespace to vLLM on port 8000, so opening this path does not open the cluster.
- **CRD field names drift.** This module targets `kagent.dev/v1alpha2`. If an apply is rejected, `kubectl explain modelconfig.spec` and `kubectl explain agent.spec` show the names your installed CRDs expect.

> NOTE: The placeholder Secret holds `not-needed` because vLLM does not enforce an API key. If you put auth in front of your vLLM, replace that value with the real key and the same `ModelConfig` keeps working.

## Try it yourself

**Ask it what powers it.** Invoke the agent with "What model and endpoint are you running on?" and confirm it names your model and your vLLM URL. That is the agent reporting, in its own words, that it runs on inference you own.

**Tighten the scope.** Edit the system prompt to add or remove an in-scope area, re-apply the Agent, and re-test. Watch the routing change on the next invoke. **Stretch:** add a real tool through a `RemoteMCPServer` and give the agent hands as well as a brain.

**Bridge it to your phone with Discord (take-home).** A small bot listens for messages, forwards them to the agent, and posts replies back. The ready-made pattern is the [nba-discord-agent](https://github.com/labeveryday/nba-discord-agent): reuse its event loop, command handling, and long-message chunking, and swap the backend call to hit your kagent agent. A Discord bot per student needs a bot token and a Discord app, which is real setup tax in a live room, so treat this as the take-home extension. The bridge is glue, not the lesson. The agent already runs and answers on inference you own; Discord just gives it your phone.

In [ ]:
# Change the task, then run the cell. Requires the running agent from section 7.
# pf must still be active; if not, re-run section 7 first.
ask("What model and endpoint are you running on?")

## Summary

- An agent is a client of a model. The whole point of this module is that its model is the vLLM you own, on your GPU, not a rented API.
- kagent runs agents as Kubernetes resources. A `ModelConfig` with the OpenAI provider points at your vLLM; an `Agent` carries the persona and references it.
- The agent lives in the `kagent` namespace, so it reaches your vLLM by the cross-namespace FQDN `vllm.<your-namespace>.svc.cluster.local`, allowed by one `NetworkPolicy` rule.
- vLLM must be served with tool calling enabled, because kagent always sends tool-calling requests.
- You invoked the agent in scope and out of scope and watched it answer and route honestly, every turn served by the inference you tuned.

## Done

You started by renting inference and ended with an agent answering on a GPU you tuned yourself. You own the whole stack now: the model, the server, the metrics, the tuning, and the agent on top of it.